In [ ]:
#| default_exp lemma

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import re
from fastcore.all import AttrDict, L, Path, ifnone, patch
from fastlite import Database
from litesearch.sanskrit import fold_token, _DEVA
from ganapati.text import VerseChunker, ProseChunker, sanskrit_parse, is_sanskrit, _detag
from ganapati.metre import metrical_text, verse_meta

## What the fold cannot do

`fold_token` makes two spellings of the same word collide. It cannot make two *forms* of a word
collide: sandhi and inflection mean a reader who types `gam` will not find `gacchati`. A lemma
does that, and a lemma needs a dictionary, which is what vidyut is.

So this is opt-in. Without vidyut, a store still gets metre. With it, a store also gets lemmas
and Monier-Williams glosses.

In [ ]:
#| export
_VIDYUT_URL_NOTE = 'https://github.com/ambuda-org/vidyut'   # MIT
MW_URL = 'https://raw.githubusercontent.com/sanskrit-lexicon/csl-orig/master/v02/mw/mw.txt'

def sanskrit_home(name:str=None) -> Path:
    'Where the downloaded Sanskrit data lives: `$LITESEARCH_HOME`, else the XDG cache.'
    import os
    d = Path(os.environ.get('LITESEARCH_HOME')
             or Path(os.environ.get('XDG_CACHE_HOME', Path.home()/'.cache'))/'litesearch')/'sanskrit'
    d.mkdir(parents=True, exist_ok=True)
    return d/name if name else d

def vidyut_data(path=None, force:bool=False) -> Path:
    "vidyut's linguistic data, downloaded on first use (~81 MB) and cached thereafter."
    d = Path(path) if path else sanskrit_home('vidyut')
    if force or not (d/'kosha').exists():
        try: import vidyut
        except ImportError: raise ImportError('lemmas and glosses need vidyut: pip install vidyut') from None
        d.mkdir(parents=True, exist_ok=True)
        vidyut.download_data(str(d))
    return d

def to_slp1(text:str) -> str:
    'Any supported script to SLP1, which is what the kosha is keyed on. Needs no downloaded data.'
    from vidyut.lipi import transliterate, Scheme
    return transliterate(text, Scheme.Devanagari if _DEVA.search(text or '') else Scheme.Iast, Scheme.Slp1)

def from_slp1(text:str, iast:bool=True) -> str:
    'SLP1 back to something readable — IAST by default, Devanagari otherwise.'
    from vidyut.lipi import transliterate, Scheme
    return transliterate(text or '', Scheme.Slp1, Scheme.Iast if iast else Scheme.Devanagari)

class _VTok:
    'A spaCy-shaped token, so `lemma_facets` needs no knowledge of where the lemma came from.'
    is_punct = is_digit = False
    def __init__(self, text, lemma, slp=''): self.text, self.lemma_, self.lemma_slp = text, lemma, slp
    def __repr__(self): return f'_VTok({self.text!r}→{self.lemma_!r})'

_TOKEN = re.compile(r'[^\s।॥|/,;:()\[\]]+')

def _pausa_keys(slp:str) -> L:
    'The forms to try in the kosha for one written word, best first.'
    ks = L(slp)
    if slp.endswith('H'): ks += [slp[:-1]+'s', slp[:-1]+'r']
    if slp.endswith('M'): ks.append(slp[:-1]+'m')
    return ks

def _splitter(path=None):
    'vidyut\'s sandhi rule table, as a splitter. Cached by the caller.'
    from vidyut.sandhi import Splitter
    return Splitter.from_csv(str(vidyut_data(path)/'sandhi'/'rules.csv'))

def _cover(w:str, sp, known, depth:int=0, max_depth:int=3, min_piece:int=2) -> tuple:
    'Split an SLP1 string into pieces the kosha recognises, or `()` if it cannot be covered.'
    if known(w): return (w,)
    # SLP1 is an ASCII scheme, and vidyut's splitter indexes by *byte*. A character `to_slp1` could
    # not map survives into the string, and `split_at` then panics from Rust — not an exception
    # Python can catch. Real editions carry them: a nukta in `पितṛ़न` (Gītā 1.34) is enough.
    if not w.isascii(): return ()
    if depth >= max_depth or len(w) < 2*min_piece: return ()
    for i in range(len(w)-min_piece, min_piece-1, -1):
        for s in sp.split_at(w, i):
            if not s.is_valid or len(s.first) < min_piece or not known(s.first): continue
            if (rest := _cover(s.second, sp, known, depth+1, max_depth, min_piece)): return (s.first,)+rest
    return ()

def vidyut_pipe(path=None, split:bool=True):
    "A spaCy-shaped lemmatiser backed by vidyut's kosha — the deterministic answer to inflection."
    from vidyut.kosha import Kosha
    ko = Kosha(str(vidyut_data(path)/'kosha'))
    sp = _splitter(path) if split else None
    def lem(k):
        try: return dict.fromkeys(e.lemma for e in ko.get(k) if e.lemma)
        except Exception: return {}
    known = lambda s: bool(s) and any(ko.get(k) for k in _pausa_keys(s))
    def pipe(text:str):
        out = []
        for w in _TOKEN.findall(text or ''):
            w = w.strip("'’॒॑")
            if not w or not (_DEVA.search(w) or _IAST_DIAC.search(w) or w.isalpha()): continue
            slps, slp = (), to_slp1(w)
            for k in _pausa_keys(slp):
                if (slps := lem(k)): break
            # nothing in the lexicon: the token is probably two words fused by sandhi
            if not slps and sp is not None and len(slp) >= 5:
                for piece in _cover(slp, sp, known):
                    for k in _pausa_keys(piece):
                        if (pl := lem(k)): slps = {**slps, **pl}; break
            out += [_VTok(w, from_slp1(s), s) for s in slps]
        return out
    return pipe

def sanskrit_terms(path=None,        # vidyut data dir; None -> `sanskrit_home()`
                   topk:int=12,      # most frequent nominals kept per chunk
                   min_len:int=3,    # skip particles and one-syllable words
                   split:bool=True   # undo sandhi on tokens the kosha does not know
                   ):
    'A `terms_fn` for `build_graph`: the nominal words of a Sanskrit chunk, via the kosha.'
    from vidyut.kosha import Kosha, PadaEntry
    ko = Kosha(str(vidyut_data(path)/'kosha'))
    sp = _splitter(path) if split else None
    known = lambda s: bool(s) and any(ko.get(k) for k in _pausa_keys(s))
    def entries(slp):
        for k in _pausa_keys(slp):
            if (es := ko.get(k)): return es
        return []
    def nominal(w) -> bool:
        es = entries(to_slp1(w))
        if not es and sp is not None and len(w) >= 4:
            # a fused token counts as nominal only if every piece of it is
            parts = _cover(to_slp1(w), sp, known)
            if not parts: return False
            return all(nominal_slp(p) for p in parts)
        return bool(es) and not any(isinstance(e, PadaEntry.Tinanta) for e in es)
    def nominal_slp(slp) -> bool:
        es = entries(slp)
        return bool(es) and not any(isinstance(e, PadaEntry.Tinanta) for e in es)
    def terms(text:str, topk:int=topk) -> L:
        seen = {}
        for w in _TOKEN.findall(metrical_text(text) or ''):
            w = w.strip("'’॒॑")
            if len(w) < min_len or not (_DEVA.search(w) or _IAST_DIAC.search(w) or w.isalpha()): continue
            if w in seen: seen[w] += 1; continue
            if nominal(w): seen[w] = 1
        return L(sorted(seen, key=lambda k: -seen[k])[:topk])
    return terms

_MW_DROP = re.compile(r'<(s|s1|ls|lex|ab|hom|info|srs|etym|gk|lang|bot|pb|div)[^>]*>.*?</\1>'
                      r'|<(info|srs|pb|div|s1)[^>]*/?>', re.S)
_MW_TAG, _MW_K1 = re.compile(r'<[^>]+>'), re.compile(r'<k1>([^<]*)')
_MW_WORD = re.compile(r'[A-Za-z][A-Za-z-]{2,}')

def _mw_clean(body:str) -> str:
    t = body.split('¦', 1)[-1] if '¦' in body else body
    t = _MW_TAG.sub(' ', _MW_DROP.sub(' ', t)).replace('&c.', '').replace('√', ' ')
    t = re.sub(r'\([^)]*\)', ' ', t)                 # parentheticals are citations or grammar
    t = re.sub(r'[\[\]‘’"]', ' ', t)
    # A root entry opens with its conjugation table, which is all `<s>`/`<ab>` and strips to bare
    # punctuation — `gam` came out as `1. ; 2. ( ; 3. ,` before this line.
    if not _MW_WORD.search(t.split(',')[0] or ''): t = re.sub(r'(?:\s*[\d.,;:—-]+\s*)+', ' ', t)
    t = re.sub(r'\s+', ' ', t).strip(' ,;.:—-')
    if (m := re.search(r'[A-Za-z]', t)): t = t[m.start():]
    return re.sub(r'\s+', ' ', t).strip(' ,;.:—-')

def _mw_ok(g:str) -> bool:
    'Real English, not the debris a conjugation table leaves behind.'
    return len(_MW_WORD.findall(g)) >= 2 and len(re.sub(r'[^A-Za-z]', '', g))/max(len(g), 1) > 0.55

def _clip(s:str, n:int) -> str:
    'Truncate on a word boundary — a half word like `ema` is an index term that matches nothing.'
    return s if len(s) <= n else s[:n].rsplit(' ', 1)[0].rstrip(' ,;')

def mw_lexicon(path=None, force:bool=False, maxlen:int=110) -> dict:
    'Monier-Williams as `{SLP1 headword: short English gloss}`, downloaded and reduced on first use.'
    tsv = (Path(path) if path else sanskrit_home())/'mw.tsv'
    if force or not tsv.exists():
        import urllib.request
        raw = tsv.with_suffix('.src')
        if force or not raw.exists():
            req = urllib.request.Request(MW_URL, headers={'User-Agent': 'litesearch'})
            with urllib.request.urlopen(req, timeout=300) as r, open(raw, 'wb') as f:
                while (b := r.read(1 << 20)): f.write(b)
        out, k, buf = {}, None, []
        with open(raw, encoding='utf-8', errors='replace') as f:
            for ln in f:
                if ln.startswith('<L>'): m = _MW_K1.search(ln); k, buf = (m.group(1) if m else None), []
                elif ln.startswith('<LEND>'):
                    if k and buf and _mw_ok(g := _mw_clean(' '.join(buf))):
                        prev = out.get(k)
                        out[k] = _clip(g, maxlen) if not prev else (
                            _clip(prev+'; '+g, maxlen) if len(prev) < 60 else prev)
                    k, buf = None, []
                elif k is not None: buf.append(ln.rstrip('\n'))
        tsv.write_text(''.join(f'{a}\t{b}\n' for a, b in out.items()), encoding='utf-8')
        raw.unlink(missing_ok=True)
    return dict(ln.split('\t', 1) for ln in tsv.read_text(encoding='utf-8').splitlines() if '\t' in ln)

GLOSS_STOP = frozenset('''the and for with that this from any all not are was has had its his her
their they which who whom what when where how such into out off over under also more most other
some one two being been have does did must may might can could would should shall will upon than
then them these those there here about above after before between during through against among
substitution used said say saying called esp cf viz ibid etc set out come towards'''.split())

def gloss_facets(text:str,          # chunk text
                 nlp,               # a `vidyut_pipe()`; tokens must carry `.lemma_slp`
                 mw:dict,           # a `mw_lexicon()`
                 max_terms:int=24   # cap, so the gloss cannot outgrow the verse
                 ) -> dict:
    "`{'gloss': 'law duty; land soil; ...'}` — the English behind a chunk's Sanskrit words."
    if not (nlp and mw and (text or '').strip()): return {}
    out = {}
    for t in nlp(metrical_text(text)):
        if (g := mw.get(getattr(t, 'lemma_slp', ''))):
            for w in g.replace(';', ' ').replace(',', ' ').split():
                w = w.lower()
                if len(w) > 2 and w.isalpha() and w not in GLOSS_STOP: out[w] = None
        if len(out) >= max_terms: break
    return {'gloss': ' '.join(list(out)[:max_terms])} if out else {}

## Facets

`lemma_facets` keeps only the lemmas the surface index does not already hold: if `fold_token`
maps the lemma and the surface form to the same key, the lemma adds nothing and is dropped. The
cap keeps a chunk's metadata from outgrowing its content.

`sanskrit_meta` is what a profile calls. With no pipeline it is `verse_meta` itself, the same
object, so the metre-only path costs nothing.

In [ ]:
#| export
def lemma_facets(text:str,          # chunk text
                 nlp,               # a `vidyut_pipe()` (or any spaCy-shaped pipeline)
                 max_terms:int=96   # cap, so one chunk's metadata cannot outgrow its content
                 ) -> dict:
    "`{'lemma': 'gam vac ...'}` for a chunk — the dictionary forms behind its inflected words."
    if not (nlp and (text or '').strip()): return {}
    try: doc = nlp(metrical_text(text))
    except Exception: return {}
    out = {}
    for t in doc:
        lem, surf = (getattr(t, 'lemma_', '') or '').strip(), (t.text or '').strip()
        if not lem or getattr(t, 'is_punct', False) or getattr(t, 'is_digit', False): continue
        if fold_token(lem) == fold_token(surf): continue     # nothing the surface index lacks
        if len(lem) < 2: continue
        out[lem] = None
        if len(out) >= max_terms: break
    return {'lemma': ' '.join(out)} if out else {}

def sanskrit_meta(nlp=None, mw:dict=None):
    'The `Profile.meta` callable: metre always, lemmas and glosses when their sources are supplied.'
    if nlp is None: return verse_meta
    if mw is None:
        def meta(text:str) -> dict: return {**verse_meta(text), **lemma_facets(text, nlp)}
    else:
        def meta(text:str) -> dict:
            return {**verse_meta(text), **lemma_facets(text, nlp), **gloss_facets(text, nlp, mw)}
    return meta

@patch
def by_lemma(self:Database,
             lemma:str,           # a dictionary form, e.g. `gam`
             store:str='store',
             prefix:str=None,
             columns:list=None,
             limit:int=50) -> list:
    'Chunks whose verses inflect `lemma`. Needs an index built with `register_profiles(nlp=...)`.'
    cols = list(dict.fromkeys((columns or ['content']) + ['metadata', 'node_id', 'doc_id', 'page']))
    return self.t[ifnone(prefix, '') + store](
        select=', '.join(cols), where='metadata like :sa_l',
        where_args={'sa_l': f'%"lemma": "%{lemma}%'}, limit=limit)

@patch
def by_meter(self:Database,
             meter:str=None,      # metre name, e.g. `mandākrāntā`
             gana:str=None,       # gaṇa signature, `ma bha na ta ta ga ga` or `ma_bha_na_...`
             store:str='store',
             prefix:str=None,
             columns:list=None,
             limit:int=50) -> list:
    'Chunks whose verses are in a metre, or share a gaṇa signature. A filter, not a ranking.'
    wh, wa = [], {}
    if meter: wh.append('metadata like :sa_m'); wa['sa_m'] = f'%"meter": "%{meter}%'
    if gana:  wh.append('metadata like :sa_g'); wa['sa_g'] = f'%{gana.strip().replace(" ", "_")}%'
    if not wh: return []
    cols = list(dict.fromkeys((columns or ['content']) + ['metadata', 'node_id', 'doc_id', 'page']))
    return self.t[ifnone(prefix, '') + store](select=', '.join(cols), where=' AND '.join(wh),
                                              where_args=wa, limit=limit)

## Searching what was written

`by_lemma` and `by_meter` filter on the metadata the facets wrote. Both are filters and not
rankings: they narrow, and hybrid search orders what is left.

## Profiles

`register_profiles()` tells litesearch how to read a Sanskrit file: which extensions to try, how
to sniff one that shares an extension with everything else, which chunker, and what metadata to
write. It runs on import, so `import ganapati` is all a caller does.

Call it again with `nlp=` and `mw=` to add lemmas and glosses to everything indexed after.

In [ ]:
#| export
def _sniff(text:str) -> bool:
    'Whether a shared extension (`.xml`, `.txt`, `.htm`) holds Sanskrit this module can read.'
    raw = (text or '')[:60000]
    if '<lyrics' in raw[:4000]: return True
    # TEI states its language, which beats sniffing: a 16 KB edition can be almost entirely header,
    # and TEI keeps the citation in an `xml:id` attribute where no content sniff will find it.
    if ('tei-c.org' in raw or '<teiHeader' in raw) and re.search(r'xml:lang=["\'](sa|pi|pra)\b', raw): return True
    t = _detag(raw)
    if re.search(r'^#\s*id\s*=', t, re.M): return True
    return is_sanskrit(t)

def register_profiles(nlp=None, mw:dict=None):
    'Register the Sanskrit profiles. Called on import; safe to call again.'
    from litesearch.data import Profile, register_profile
    meta = sanskrit_meta(nlp, mw)
    register_profile(Profile(name='sanskrit_verse', exts='.xml,.tei,.htm,.html,.conllu,.txt',
                             parse=sanskrit_parse, chunker=VerseChunker, mode='verse',
                             detect=_sniff, kind='sanskrit', meta=meta))
    register_profile(Profile(name='sanskrit_prose', exts='', parse=sanskrit_parse,
                             chunker=ProseChunker, mode='verse',
                             detect=lambda _t: False, kind='sanskrit', meta=meta))

register_profiles()

In [ ]:
from litesearch.data import profile_for
import ganapati

# a GRETIL file now routes to the verse profile rather than to plain text. The sniff wants
# three citations before it commits, so one line is not enough to identify a file.
gretil = "atha prathamo'dhyāyaḥ || MBh_1,1.1 ||\n" * 10
p = profile_for('mahabharata_u.htm', gretil)
assert p and p.name == 'sanskrit_verse'
assert p.chunker is VerseChunker and p.kind == 'sanskrit'

# with no pipeline the metadata callable is verse_meta itself, not a wrapper around it
from ganapati.lemma import sanskrit_meta
from ganapati.metre import verse_meta
assert sanskrit_meta(None) is verse_meta
# and it is still the metre callable: a real verse in, its metre out
assert sanskrit_meta(None)('dharmakṣetre kurukṣetre samavetā yuyutsavaḥ '
                          'māmakāḥ pāṇḍavāścaiva kimakurvata sañjaya')['meter'] == 'anuṣṭubh'
assert sanskrit_meta(None)('') == {}

In [ ]:
#| eval: false
# vidyut is an 81 MB download, so this runs only where the data is already cached.
from ganapati.lemma import vidyut_pipe, lemma_facets
nlp = vidyut_pipe()
f = lemma_facets('gacchati rāmaḥ', nlp)
assert 'gam' in f['lemma']         # the dictionary form the surface index does not hold

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()